# Baseline Comparison — HTD detectors

Pluggable framework. Each detector lives in its own dir and never knows whether its data is spatial / iid / synthetic. Two tracks:

- **EASY** — low-dimensional synthetic GMM (fast). i.i.d. and spatially-correlated. Sweeps target *model* (additive + **full replacement**), *signature* (comp_mean / between / orthogonal) and *amplitude*. Full-replacement (θ=1) is the *did-it-train* check.
- **HARD** — real Pavia scenarios (additive, amplitude sweep around 0.15).

Everything is saved per detector (model, scores, maps, metrics, train_log, config) so figures are remade WITHOUT retraining.

In [ ]:
# ── Cell 1: Clone/pull (branch iid-unified-experiment) + deps ─────────────
import os, sys
REPO_URL='https://github.com/michaelpiro/final-paper-experiment.git'
LOCAL='/content/proj'; BRANCH='iid-unified-experiment'   # has all the latest code
if os.path.exists(os.path.join(LOCAL,'.git')):
    !git -C {LOCAL} fetch --all -q
    !git -C {LOCAL} checkout {BRANCH}
    !git -C {LOCAL} pull origin {BRANCH}
else:
    !git clone -b {BRANCH} {REPO_URL} {LOCAL}
!pip install -q scikit-learn scipy tqdm matplotlib pyyaml
for p in [LOCAL, os.path.join(LOCAL,'experiments','spatial')]:
    if p not in sys.path: sys.path.insert(0,p)
os.chdir(LOCAL); print('CWD', os.getcwd())
!git -C {LOCAL} log --oneline -1

In [ ]:
# ── Cell 2: Mount Drive (data + results) — only needed for the HARD/Pavia run ──
from google.colab import drive; drive.mount('/content/drive')
import os
os.makedirs('/content/proj/data', exist_ok=True)
DST='/content/proj/data/pavia-u.mat'
if not os.path.exists(DST):
    for src in ['/content/drive/MyDrive/final_paper/pavia-u.mat',
                '/content/drive/MyDrive/final_paper/real_datasets/pavia-u.mat']:
        if os.path.exists(src): os.symlink(src, DST); print('linked', src); break
RESULTS='/content/drive/MyDrive/final_paper/baseline_results'; os.makedirs(RESULTS, exist_ok=True)
print('RESULTS', RESULTS)

## EASY case — synthetic GMM (low-dim, fast)

In [ ]:
# ── Cell E1: EASY dry-run (quick sanity, both providers) ──────────────────
RESULTS='/content/drive/MyDrive/final_paper/baseline_results'
CFG='experiments/baseline_comparison/configs/synthetic_gmm.yaml'
!python -u experiments/baseline_comparison/run_comparison.py --config {CFG} --provider spatial_gmm --results_dir {RESULTS} --dry-run
!python -u experiments/baseline_comparison/run_comparison.py --config {CFG} --provider iid_gmm     --results_dir {RESULTS} --dry-run

In [ ]:
# ── Cell E2: EASY full run (low-dim => fast even on CPU) ───────────────────
RESULTS='/content/drive/MyDrive/final_paper/baseline_results'
CFG='experiments/baseline_comparison/configs/synthetic_gmm.yaml'
!python -u experiments/baseline_comparison/run_comparison.py --config {CFG} --provider spatial_gmm --results_dir {RESULTS}
!python -u experiments/baseline_comparison/run_comparison.py --config {CFG} --provider iid_gmm     --results_dir {RESULTS}

In [ ]:
# ── Cell E3: EASY figures (edit freely) ───────────────────────────────────
import glob, matplotlib.pyplot as plt
from experiments.baseline_comparison.framework import figures as F
RESULTS='/content/drive/MyDrive/final_paper/baseline_results'
RUN = sorted(glob.glob(f'{RESULTS}/spatial_gmm_*'))[-1]   # latest spatial-GMM run
sc = F.scenarios(RUN)[0]
print('RUN', RUN, '| scenario', sc, '| detectors', F.detectors(RUN, sc))
F.summary_table(RUN, sc, 'additive', 'orthogonal')
for sig in ['comp_mean','between','orthogonal']:
    F.auc_vs_amplitude(RUN, sc, 'additive', sig); plt.show()
F.replacement_sanity(RUN, sc, 'orthogonal', 1.0); plt.show()   # did-it-train check
F.roc(RUN, sc, 'additive', 'orthogonal'); plt.show()
F.detection_maps(RUN, sc, 'additive', 'orthogonal'); plt.show()

## HARD case — Pavia scenarios (additive, amplitude sweep)

In [ ]:
# ── Cell H1: HARD dry-run ─────────────────────────────────────────────────
RESULTS='/content/drive/MyDrive/final_paper/baseline_results'
CFG='experiments/baseline_comparison/configs/pavia_scenarios.yaml'
!python -u experiments/baseline_comparison/run_comparison.py --config {CFG} --results_dir {RESULTS} --dry-run

In [ ]:
# ── Cell H2: HARD full run (Pavia, all detectors, amplitude sweep) ────────
RESULTS='/content/drive/MyDrive/final_paper/baseline_results'
CFG='experiments/baseline_comparison/configs/pavia_scenarios.yaml'
!python -u experiments/baseline_comparison/run_comparison.py --config {CFG} --results_dir {RESULTS}

In [ ]:
# ── Cell H3: HARD figures (edit freely) ───────────────────────────────────
import glob, matplotlib.pyplot as plt
from experiments.baseline_comparison.framework import figures as F
RESULTS='/content/drive/MyDrive/final_paper/baseline_results'
RUN = sorted(glob.glob(f'{RESULTS}/pavia_*'))[-1]
for sc in F.scenarios(RUN):
    print('='*60); F.summary_table(RUN, sc, 'additive')
    F.auc_vs_amplitude(RUN, sc, 'additive'); plt.show()
    F.roc(RUN, sc, 'additive', amplitude=0.15); plt.show()
    F.detection_maps(RUN, sc, 'additive', amplitude=0.15); plt.show()